In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns

from data_profiling import ProfileReport

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

Regresión Supervisada

In [ ]:
df = pd.read_csv('data/house_pricing.csv', sep=',')

In [ ]:
df.describe()

Elimino los datos nulos

In [ ]:
#pred = df.dropna(axis=0)

Comprensión de los datos


Fase 2: Comprensión de los Datos (Data Understanding)
Basado en la metodología CRISP-DM y las directrices de inspección de calidad de datos.

In [ ]:
##%% md
# # Fase 2: Comprensión de los Datos (Data Understanding)
# Basado en la metodología CRISP-DM y las directrices de inspección de calidad de datos.

# Configuración estética de los gráficos
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# 1. Carga del conjunto de datos original
df = pd.read_csv('data/house_pricing.csv', sep=',')

##%% md
# ## 2.1. Primera Inspección Básica (Estructura y Tipos)
# Verificación de las dimensiones del dataset y consistencia en los tipos de datos asignados por Pandas.

##%%
print(f"=== Dimensiones del Dataset ===")
print(f"Filas (Instancias): {df.shape[0]}")
print(f"Columnas (Atributos): {df.shape[1]}\n")

print("=== Primeras Filas del Dataset ===")
display(df.head())

print("\n=== Distribución de Tipos de Datos ===")
print(df.dtypes.value_counts())

##%% md
# ## 2.2. Exploración de la Variable Objetivo (SalePrice)
# Inspección fundamental de la distribución de los precios de venta para detectar sesgos (Skewness) 
# antes de aplicar modelos lineales.

##%%
plt.figure(figsize=(10, 5))

# Histograma con densidad (KDE)
plt.subplot(1, 2, 1)
sns.histplot(df['SalePrice'].dropna(), kde=True, color='royalblue')
plt.title('Distribución Original de SalePrice')
plt.xlabel('Precio de Venta ($)')

# Transformación logarítmica sugerida si hay fuerte asimetría
plt.subplot(1, 2, 2)
sns.histplot(np.log1p(df['SalePrice'].dropna()), kde=True, color='forestgreen')
plt.title('Distribución Logarítmica de SalePrice')
plt.xlabel('Log(Precio de Venta + 1)')

plt.tight_layout()
plt.show()

# Imprimir asimetría numérica
print(f"Asimetría (Skewness) original: {df['SalePrice'].skew():.2f}")

##%% md
# ## 2.3. Verificación de la Calidad de los Datos (Errores y Nulos)
# Identificación de registros duplicados y análisis porcentual profundo de los valores ausentes.

##%%
# 1. Filas completamente duplicadas
duplicados = df.duplicated().sum()
print(f"Total de filas completamente duplicadas: {duplicados}")

# 2. Análisis detallado de Valores Faltantes (Missing Values)
valores_nulos = df.isna().sum()
nulos_filtrados = valores_nulos[valores_nulos > 0].sort_values(ascending=False)

if not nulos_filtrados.empty:
    df_nulos = pd.DataFrame({
        'Total Nulos': nulos_filtrados,
        'Porcentaje (%)': (nulos_filtrados / len(df)) * 100
    })
    print("\n=== Atributos con valores ausentes ===")
    display(df_nulos)
else:
    print("\n¡Excelente! No se detectaron valores nulos en el dataset.")

##%% md
# ## 2.4. Análisis Estadístico y Correlación Lineal
# Inspección de las estadísticas descriptivas y detección de colinealidad con la variable objetivo.

##%%
print("=== Estadísticas Descriptivas de Variables Numéricas ===")
display(df.describe().T)

# Matriz de correlación lineal con SalePrice (solo variables numéricas relevantes)
print("\n=== Top 10 Variables más correlacionadas con SalePrice ===")
correlaciones = df.select_dtypes(include=[np.number]).corr()['SalePrice'].sort_values(ascending=False)
print(correlaciones.head(11)) # Muestra las 10 más SalePrice misma

##%% md
# ## 2.5. Exploración Automática (Data Profiling Avanzado)
# Generación de informe interactivo HTML optimizado eliminando interacciones pesadas 
# para evitar sobrecarga de memoria del kernel.

##%%
# Configurar el reporte automático adaptado utilizando la librería 'data_profiling'
profile = ProfileReport(
    df,
    title="Informe de Exploración Automática - Ames Housing",
    explorative=True,
    interactions={
        "continuous": False  # Crucial desactivarlo para optimizar el rendimiento de la memoria
    },
    correlations={
        "pearson": {"calculate": True, "threshold": 0.85},
        "spearman": {"calculate": False},
        "kendall": {"calculate": False},
        "phi_k": {"calculate": False},
        "cramers": {"calculate": False},
        "auto": {"calculate": False, "warn_high_correlations": False}
    }
)

# Creación de directorio y almacenamiento físico del reporte
os.makedirs('reports', exist_ok=True)
ruta_reporte = "reports/house_pricing_report.html"
profile.to_file(ruta_reporte)

print(f"\n[OK] ¡Informe automático generado y actualizado con éxito en: '{ruta_reporte}'!")

In [ ]:
df.describe().T

In [ ]:
filas_con_nulos = df[df.isna().any(axis=1)]
filas_con_nulos

In [ ]:
a = np.array([3, 5, 4, 2])
b = a[(a >= 1) | (a < 4)]
print(b)